# Run J: Qwen2.5-1.5B fine-tuned on the v6 mix (generated + real CoCoLoFa), then scored on every v6 split
Same settings as run I (LoRA rank 32, 5 epochs, lr 2e-4, batch 8, no validation holdout). Only the training data changes: the earlier generated data plus CoCoLoFa's official train split (capped per type, label checked by an LLM, explanations written by an LLM).

**Upload to the Files sidebar and wait for each upload to finish:**
- `fallacy_code.zip` (code, tiny)
- `v6_data.zip` (training file and the calibration and test files: contains CoCoLoFa and MAFALDA text, **private, local use only**)

No Hugging Face token is needed: everything is in the zips.
Expect about 1-1.5 hours on a T4, much less on an A100 or L4 (training is most of it, then about 10-30 minutes of scoring). Keep the tab open.

In [ ]:
!nvidia-smi
!pip -q uninstall -y torchao
!pip -q install peft transformers accelerate tqdm sentencepiece
!pip list 2>/dev/null | grep -iE "^(transformers|torch|peft|accelerate) "

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
!ls -la /content/*.zip
!rm -rf /content/work && mkdir -p /content/work
!for z in /content/fallacy_code.zip /content/v6_data.zip; do unzip -tq $z || echo "BROKEN (incomplete upload): $z"; unzip -oq $z -d /content/work; done
%cd /content/work
!wc -l data/v6/train_v6.jsonl data/v6/calib/*.jsonl data/v6/test/*.jsonl

## Train (expect the line `N training examples` with N = the number of lines in train_v6.jsonl)

In [ ]:
!python -u train.py --train-files data/v6/train_v6.jsonl --out out/lora_J --epochs 5 --rank 32 --lr 2e-4 --batch 8 --val-per-label 0 2>&1 | tee train_J.log

## Score every label name on every calibration and test file (this is what the analysis needs)

In [ ]:
!python -u score_candidates.py --adapter out/lora_J --run J \
  --set calib_generated=data/v6/calib/generated.jsonl \
  --set calib_cocolofa=data/v6/calib/cocolofa.jsonl \
  --set test_generated=data/v6/test/generated.jsonl \
  --set test_cocolofa=data/v6/test/cocolofa.jsonl \
  --set test_mafalda=data/v6/test/mafalda.jsonl

In [ ]:
!zip -qr result_J.zip out/lora_J data/scores_J_*.json train_J.log -x '*/README.md'
from google.colab import files
files.download('/content/work/result_J.zip')